# Loss Functions and Optimizers

A loss function measures model error. An optimizer uses the gradients produced by `backward()` to update model parameters. The correct choices depend on the kind of problem, model output, and target format.


In [ ]:
import torch
from torch import nn

torch.manual_seed(42)


## Multiclass Classification: `CrossEntropyLoss`

Use `CrossEntropyLoss` when each sample belongs to one of several classes. Pass raw logits with shape `[batch, classes]` and integer class indices with shape `[batch]`. Do not apply softmax first because the loss handles it internally.


In [ ]:
multiclass_logits = torch.tensor([
    [2.1, 0.2, -0.4],
    [0.1, 1.4, 0.3],
])
class_targets = torch.tensor([0, 1])

cross_entropy_loss = nn.CrossEntropyLoss()(multiclass_logits, class_targets)
print(f"CrossEntropyLoss: {cross_entropy_loss.item():.4f}")


## Binary Classification: `BCEWithLogitsLoss`

Use `BCEWithLogitsLoss` for binary or multilabel classification. Logits and floating-point targets must have the same shape. This loss combines sigmoid with binary cross entropy for better numerical stability.


In [ ]:
binary_logits = torch.tensor([[1.2], [-0.7], [0.3]])
binary_targets = torch.tensor([[1.0], [0.0], [1.0]])

binary_loss = nn.BCEWithLogitsLoss()(binary_logits, binary_targets)
print(f"BCEWithLogitsLoss: {binary_loss.item():.4f}")


## Regression: `MSELoss`

Use `MSELoss` for continuous predictions. Predictions and targets should use compatible shapes; otherwise broadcasting can hide a shape mistake.


In [ ]:
predictions = torch.tensor([[2.5], [3.0], [4.5]])
regression_targets = torch.tensor([[3.0], [3.0], [5.0]])

mse_loss = nn.MSELoss()(predictions, regression_targets)
print(f"MSELoss: {mse_loss.item():.4f}")


## Comparing Optimizers

- **SGD** provides direct, predictable updates and often works well with momentum.
- **Adam** adapts the learning rate for each parameter and is a common starting point.
- **AdamW** decouples weight decay from the gradient update and is preferred when using Adam with regularization.

For a fair comparison, each optimizer trains the same model from the same initial weights and on the same data.


In [ ]:
regression_inputs = torch.linspace(-1, 1, 64).unsqueeze(1)
regression_labels = 3 * regression_inputs + 0.5

def train_with(optimizer_type, learning_rate, weight_decay=0.0):
    torch.manual_seed(42)
    model = nn.Linear(1, 1)
    optimizer = optimizer_type(
        model.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay,
    )
    loss_fn = nn.MSELoss()

    for _ in range(100):
        optimizer.zero_grad()
        loss = loss_fn(model(regression_inputs), regression_labels)
        loss.backward()
        optimizer.step()

    return loss.item()


### Run the Comparison

The learning rates below are selected independently because optimizer learning-rate scales are not directly interchangeable. Compare the final loss while remembering that real experiments should also compare validation performance.


In [ ]:
optimizer_results = {
    "SGD": train_with(torch.optim.SGD, learning_rate=0.1),
    "Adam": train_with(torch.optim.Adam, learning_rate=0.05),
    "AdamW": train_with(
        torch.optim.AdamW,
        learning_rate=0.05,
        weight_decay=0.01,
    ),
}

for optimizer_name, final_loss in optimizer_results.items():
    print(f"{optimizer_name:5} final loss: {final_loss:.6f}")
